# 30 — Exposure, reassociation and count controls (revision round 1, reviewer R2-2)

Same inputs, ranking and locked trainer as kernel 27. New arm paste_real and k = 11, 22, 44. All arms in one session. Validation only; the test split is not read.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
WORK = Path('/kaggle/working/exposure_controls_v1')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
# Kaggle unpacks an uploaded zip into plain files; accept either form, never both.
archives = list(INPUT.rglob('texture_study_code.zip'))
# A previous stage's kernel output also carries a code copy next to study/; only the code dataset counts.
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json')
            if (p.parent/'code/cyclegan/tools/texture_study.py').exists() and not (p.parent/'study').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)  # /kaggle/input is read-only
import hashlib
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
ENV = {**os.environ, 'PYTHONPATH':str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'PYTHONUNBUFFERED':'1'}
def unique(marker):
    hits=sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits)!=1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
ASSETS = unique('reviewed_masks.json')
ENV.update(MELON_DATASET=str(DATASET),MELON_ASSETS=str(ASSETS))
os.environ.update(ENV)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=ENV,check=True)
CLI = HERE/'tools/texture_study.py'
import torch
print(torch.__version__,torch.cuda.is_available())


In [ ]:
# ultralytics pinned to the version of the balancing experiment (C) so the trainer is identical.
import importlib.util, importlib.metadata as meta
if importlib.util.find_spec('ultralytics') is None or meta.version('ultralytics') != '8.4.163':
    subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.163'],check=True)
for package in ('torch','torchvision','ultralytics'): print(package, meta.version(package))
if torch.cuda.device_count() < 1: raise RuntimeError('GPU required')
print('GPUs', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


In [ ]:
# R512 pool: same crop windows, same reviewed-pool membership and same GrabCut masks (nearest-upsampled),
# images re-cropped from the original photographs at 512 px. Only resolution changes.
def build_pool512(src, dst, dataset, size=512, require_train=True):
    import csv, json, collections, hashlib
    from pathlib import Path
    import numpy as np, cv2
    from PIL import Image
    from melon_cyclegan import config as cfg
    from melon_cyclegan.crops import crop_with_reflection
    from melon_cyclegan.color import phi_numpy
    from melon_cyclegan.training_data import digest, verify, sha
    src, dst, dataset = Path(src), Path(dst), Path(dataset)
    if dst.exists(): raise ValueError(f'{dst} exists')
    man = json.loads((src/'crops_manifest.json').read_text())
    rows = {r['id']: r for r in csv.DictReader(open(dataset/'manifest.csv'))}
    k = size/float(man.get('size', 256)); records=[]; feats=collections.defaultdict(list); drift=[]
    for r in man['records']:
        row = rows[r['id']]
        if (require_train and row['split']!='train') or sha(dataset/row['image'])!=r['source_sha256']: raise ValueError(f"{r['id']}: source photo differs")
        with Image.open(dataset/row['image']) as h: img=h.convert('RGB')
        left, top, side = r['crop_window']
        crop = cv2.resize(crop_with_reflection(img,left,top,side),(size,size),interpolation=cv2.INTER_AREA)
        m256 = Image.open(src/r['mask']).convert('L')
        mask = np.asarray(m256.resize((size,size),Image.Resampling.NEAREST))
        old = np.asarray(Image.open(src/r['image']).convert('RGB'),np.float64)
        drift.append(np.abs(cv2.resize(crop,old.shape[1::-1],interpolation=cv2.INTER_AREA).astype(np.float64)-old).mean())
        rec = dict(r, crop_box=[float(v)*k for v in r['crop_box']])
        for key, arr in (('image', crop), ('mask', mask)):
            p = dst/r[key]; p.parent.mkdir(parents=True, exist_ok=True); Image.fromarray(arr).save(p); rec[f'{key}_sha256']=sha(p)
        feats[int(r['class_id'])].append(phi_numpy(crop.astype(np.float64)/255.0, mask>127)); records.append(rec)
    if max(drift) > 4.0: raise ValueError(f'512 crops do not match the 256 pool geometry (max mean abs diff {max(drift):.2f})')
    classes = {}
    for cid, v in sorted(feats.items()):
        x=np.stack(v); classes[cfg.CLASS_NAMES[cid]]=dict(class_id=cid,count=len(x),centroid=x.mean(0).tolist(),spread=x.std(0).tolist())
    new = {k2: v for k2, v in man.items() if k2 not in ('records','crops_digest','centroids_sha256')}
    new.update(records=records, size=size, crops_digest=digest(records), source_crops_digest_256=man['crops_digest'],
               note='R512: same windows, members and GrabCut masks as the 256 pool; images re-cropped at 512')
    (dst/'color_centroids.json').write_text(json.dumps(dict(version=cfg.VERSION, phi=man['phi'],
        source='curated train split real crops (R512)', crops_digest=new['crops_digest'], classes=classes), indent=2)+'\n')
    new['centroids_sha256'] = sha(dst/'color_centroids.json')
    (dst/'crops_manifest.json').write_text(json.dumps(new, indent=2)+'\n')
    verify(dst, man['training_policy'])
    return new, dict(max_drift=float(max(drift)), mean_drift=float(np.mean(drift)), n=len(records))


# Study (kernel 13), R512 pool (same windows/masks as kernel 25/26), and the filled blinded review.
import csv, hashlib
source=unique('study.json'); STUDY=WORK/'study'; shutil.copytree(source,STUDY)
for p in [STUDY,*STUDY.rglob('*')]: p.chmod(p.stat().st_mode|0o200)
P100=STUDY/'p100'
sys.path.insert(0,str(HERE)); sys.path.insert(0,str(HERE/'tools'))
from melon_cyclegan import texture_study as s
from melon_cyclegan import config as cfg
scope,m=s.verify_scope(STUDY,100,DATASET)
POOL512=P100/'crops512'
pool_manifest,pool_check=build_pool512(P100/'crops',POOL512,DATASET)
print('R512 pool',pool_check,pool_manifest['crops_digest'])
REV=unique('review_thumbs.npz')  # kernel 13 output also carries a review_key_PRIVATE.json
key={k['blind_id']:k for k in json.loads((REV/'review_key_PRIVATE.json').read_text())}
review=list(csv.DictReader(open(REV/'review.csv',newline='')))
if {r['blind_id'] for r in review}!=set(key) or len(review)!=len(key): raise ValueError('review rows differ from the key')
CRIT=('texture_realistic','shape_ok','boundary_ok','artifact_free')
for r in review:
    if r['accepted'] not in ('yes','no') or not r['reviewer'].strip(): raise ValueError(f"incomplete review {r['blind_id']}")
    if r['target_label'] not in ('ripe','unripe','ambiguous') or any(r[c] not in ('yes','no') for c in CRIT): raise ValueError(f"incomplete criteria {r['blind_id']}")
    if r['accepted']=='yes' and (r['target_label']!='unripe' or any(r[c]!='yes' for c in CRIT)): raise ValueError(f"accepted contradicts criteria {r['blind_id']}")
ACC={}
for r in review:
    k=key[r['blind_id']]; ACC.setdefault(k['arm'],{})[k['parent_id']]=r['accepted']=='yes'
REVIEW_SHA=hashlib.sha256((REV/'review.csv').read_bytes()).hexdigest()
print('review sha256',REVIEW_SHA,{a:f'{sum(v.values())}/{len(v)}' for a,v in ACC.items()})


In [ ]:
# Re-export the reviewed candidates as full photographs: MRT-GAN seed 12 (same EMA weights as kernel 26) and the
# colour transfer (same CIELAB affine statistics as the review images). Integrity: 64 px thumbnails of the new review
# views must match the reviewed images (mean absolute difference per image, 0-255 scale).
import numpy as np
from PIL import Image
from skimage.color import rgb2lab, lab2rgb
gbc=s.tool('generate_balance_candidates')
CAND=WORK/'candidates'
ck=[p.parent for p in INPUT.rglob('latest_net_G_B_ema.pth') if p.parent.name=='p100_G3TI_R512_seed12']
if len(ck)!=1: raise ValueError(f'expected one MRT-GAN seed-12 checkpoint, found {ck}')
gbc.run(ck[0].parent,'p100_G3TI_R512_seed12','ema',POOL512,CAND/'MRTGAN_s12',DATASET,'cuda')
st=json.loads((REV/'ct_stats.json').read_text())
muB,sdB,muA,sdA=(np.array(st[k]) for k in ('ripe_mean','ripe_sd','unripe_mean','unripe_sd'))
rows={r['id']:r for r in csv.DictReader(open(DATASET/'manifest.csv'))}
ct=CAND/'colour_transfer'
for sub in ('full/images','full/labels','images'): (ct/sub).mkdir(parents=True)
ct_records=[]
for crop in sorted(pool_manifest['records'],key=lambda r:r['id']):
    if int(crop['class_id'])!=cfg.RIPE: continue
    src=np.asarray(Image.open(POOL512/crop['image']).convert('RGB'))
    mk=np.asarray(Image.open(POOL512/crop['mask']).convert('L'))>127
    lab=rgb2lab(src/255.0); lab[mk]=(lab[mk]-muB)/sdB*sdA+muA
    out=(np.clip(lab2rgb(lab),0,1)*255).round().astype(np.uint8); out[~mk]=src[~mk]
    row=rows[crop['id']]
    with Image.open(DATASET/row['image']) as h: full=h.convert('RGB')
    synthetic,clip=gbc.paste(full,Image.fromarray(out),Image.open(POOL512/crop['mask']).convert('L'),crop['crop_window'])
    cid=f"balance__{crop['id']}"
    synthetic.save(ct/'full/images'/f'{cid}.jpg',quality=95)
    (ct/'full/labels'/f'{cid}.txt').write_text(gbc.flipped_label((DATASET/row['label']).read_text(),cfg.UNRIPE))
    l,t,sd_=crop['crop_window']; synthetic.crop((l,t,l+sd_,t+sd_)).resize((512,512),Image.Resampling.LANCZOS).save(ct/'images'/f'{cid}.png')
    ct_records.append(dict(candidate_id=cid,parent_id=crop['id'],full_image=f'full/images/{cid}.jpg',full_label=f'full/labels/{cid}.txt',image=f'images/{cid}.png'))
(ct/'records.json').write_text(json.dumps(ct_records,indent=1))
thumbs=np.load(REV/'review_thumbs.npz')
INTEGRITY={}
for arm,folder in (('MRTGAN_s12',CAND/'MRTGAN_s12'),('colour_transfer',ct)):
    diffs=[]
    for f in sorted((folder/'images').glob('balance__*.png')):
        pid=f.stem.replace('balance__','')
        new=np.asarray(Image.open(f).convert('RGB').resize((64,64),Image.Resampling.BOX)).astype(float)
        diffs.append(float(np.abs(new-thumbs[f'{arm}/{pid}']).mean()))
    INTEGRITY[arm]=dict(n=len(diffs),max=max(diffs),mean=float(np.mean(diffs)))
print('integrity',INTEGRITY)
if any(v['n']!=217 or v['max']>4.0 for v in INTEGRITY.values()): raise SystemExit('re-exported candidates differ from the reviewed images')


In [ ]:
# Rank accepted candidates by texture distance to real unripe crops, computed at 512 px with the study feature function
# (32 px patches, stride 16, coverage 0.95, first four eligible), then build four count-matched arms.
import math, torch
from melon_cyclegan.texture_study_eval import image_features
from melon_cyclegan.texture import patch_indices
METRICS=('luma_energy','chroma_energy','nyquist_luma','nyquist_r_minus_luma','nyquist_b_minus_luma')
def feats(img_path,mask_path):
    rgb=np.asarray(Image.open(img_path).convert('RGB').resize((512,512),Image.Resampling.LANCZOS)).astype(float)/255
    mk=np.asarray(Image.open(mask_path).convert('L').resize((512,512),Image.Resampling.NEAREST))>127
    ix=patch_indices(torch.from_numpy(mk.astype(np.float32))[None,None],32,16,4,.95,random=False)[0].tolist()
    cols=(512-32)//16+1
    return image_features(rgb,mk,[(i//cols*16,i%cols*16) for i in ix])
byid={r['id']:r for r in pool_manifest['records']}
ref_rows=[feats(POOL512/r['image'],POOL512/r['mask']) for r in pool_manifest['records'] if r['domain']=='A']
REF={k:float(np.mean([f[k] for f in ref_rows])) for k in METRICS}
train=[r for r in rows.values() if r['split']=='train' and r['id'] not in cfg.EXCLUDED_PARENTS]
ripe=sum(int(r['class_id'])==cfg.RIPE for r in train); unripe=sum(int(r['class_id'])==cfg.UNRIPE for r in train)
TARGET=ripe-unripe
RANK={}
for arm,folder in (('MRTGAN_s12',CAND/'MRTGAN_s12'),('colour_transfer',ct)):
    acc=[pid for pid,ok in ACC[arm].items() if ok]
    ranked=[]
    for pid in acc:
        f=feats(folder/'images'/f'balance__{pid}.png',POOL512/byid[pid]['mask'])
        ranked.append(dict(parent_id=pid,distance=sum(abs(math.log(max(f[k],1e-12)/REF[k])) for k in METRICS),**{k:f[k] for k in METRICS}))
    ranked.sort(key=lambda r:(r['distance'],s.digest(f"balance__{r['parent_id']}")))
    RANK[arm]=ranked
    print(arm,'accepted',len(acc),'deficit',TARGET)
# Count-matched additions (user decision 2026-10-04): every augmented arm adds the same K = min(deficit, accepted per synthetic arm).
K=min([TARGET]+[len(r) for r in RANK.values()])
if K<=0: raise SystemExit('no accepted candidates in a synthetic arm')
print('count-matched additions K =',K)
for arm,ranked in RANK.items():
    with open(WORK/f'ranking_{arm}.csv','w',newline='') as fh:
        w=csv.writer(fh); w.writerow(['rank','used','parent_id','distance',*METRICS])
        for i,r in enumerate(ranked): w.writerow([i+1,i<K,r['parent_id'],round(r['distance'],5),*[r[k] for k in METRICS]])
bba=s.tool('build_balance_arms'); ARMS=WORK/'arms'; allrows=list(rows.values()); excl=set(cfg.EXCLUDED_PARENTS)
ARM_INFO={}
ARM_INFO['real_only']=bba.write_arm(ARMS,'real_only',allrows,DATASET,[],excl)['counts']
order=bba.digest_order([r['id'] for r in train if int(r['class_id'])==cfg.UNRIPE])
dup=[(f"{order[i%len(order)]}__dup{i//len(order)}",DATASET/rows[order[i%len(order)]]['image'],(DATASET/rows[order[i%len(order)]]['label']).read_text(),order[i%len(order)]) for i in range(K)]
ARM_INFO[f'oversample_k{K}']=bba.write_arm(ARMS,f'oversample_k{K}',allrows,DATASET,dup,excl)['counts']
for arm,folder,name in (('colour_transfer',ct,f'colour_transfer_k{K}'),('MRTGAN_s12',CAND/'MRTGAN_s12',f'mrtgan_k{K}')):
    k=K
    add=[(f'add_{i:04d}',folder/'full/images'/f"balance__{r['parent_id']}.jpg",(folder/'full/labels'/f"balance__{r['parent_id']}.txt").read_text(),r['parent_id']) for i,r in enumerate(RANK[arm][:k])]
    if any(a[2].split()[0]!='1' for a in add): raise ValueError('synthetic labels must be unripe')
    ARM_INFO[name]=dict(bba.write_arm(ARMS,name,allrows,DATASET,add,excl)['counts'],added=k,accepted=len(RANK[arm]),full_balance=k==TARGET)
print(json.dumps(ARM_INFO,indent=1))


In [ ]:
# Real-fruit paste and count variants (protocol PROTOCOL_R1.md).

import numpy as np, cv2

def scene_mask(rec, mask512, H, W):
    l, t, s = [int(round(v)) for v in rec['crop_window']]
    m = cv2.resize(mask512, (s, s), interpolation=cv2.INTER_NEAREST); out = np.zeros((H, W), np.uint8)
    x0, y0, x1, y1 = max(0, l), max(0, t), min(W, l+s), min(H, t+s)
    out[y0:y1, x0:x1] = m[y0-t:y1-t, x0-l:x1-l]
    return out

def remove_fruit(im, mask, box):
    px = max(3, int(round(0.02*max(box[2]-box[0], box[3]-box[1]))))
    m = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*px+1, 2*px+1)))*255
    H, W = im.shape[:2]; s = 0.25
    sm = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA); mm = cv2.resize(m, (sm.shape[1], sm.shape[0]), interpolation=cv2.INTER_NEAREST)
    inp = cv2.inpaint(cv2.cvtColor(sm, cv2.COLOR_RGB2BGR), mm, 9, cv2.INPAINT_TELEA)
    up = cv2.cvtColor(cv2.resize(inp, (W, H), interpolation=cv2.INTER_CUBIC), cv2.COLOR_BGR2RGB)
    out = im.copy(); out[m > 0] = up[m > 0]; return out

def paste(crop, mask, bi, dim, dbox):
    fx0, fy0, fx1, fy1 = bi; sx, sy = (dbox[2]-dbox[0])/(fx1-fx0), (dbox[3]-dbox[1])/(fy1-fy0)
    c = cv2.resize(crop, None, fx=sx, fy=sy, interpolation=cv2.INTER_AREA if sx*sy < 1 else cv2.INTER_CUBIC)
    mk = cv2.resize(mask, (c.shape[1], c.shape[0]), interpolation=cv2.INTER_NEAREST).astype(np.float32)
    mk = cv2.GaussianBlur(mk, (0, 0), 1.5)[..., None]
    ox, oy = int(round(dbox[0]-fx0*sx)), int(round(dbox[1]-fy0*sy))
    H, W = dim.shape[:2]; out = dim.astype(np.float32).copy()
    X0, Y0 = max(0, ox), max(0, oy); X1, Y1 = min(W, ox+c.shape[1]), min(H, oy+c.shape[0])
    sub = c[Y0-oy:Y1-oy, X0-ox:X1-ox].astype(np.float32); a = mk[Y0-oy:Y1-oy, X0-ox:X1-ox]
    out[Y0:Y1, X0:X1] = a*sub+(1-a)*out[Y0:Y1, X0:X1]
    return out.round().astype(np.uint8), np.array(dbox, float)

def yolo_box(label_text, W, H):
    c, cx, cy, bw, bh = map(float, label_text.split()[:5])
    return np.array([(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H])

def to_yolo(cid, nb, W, H):
    return f"{cid} {(nb[0]+nb[2])/2/W:.6f} {(nb[1]+nb[3])/2/H:.6f} {(nb[2]-nb[0])/W:.6f} {(nb[3]-nb[1])/H:.6f}\n"

import cv2
ripe_ids=[r['id'] for r in pool_manifest['records'] if r['domain']=='B']
P=[r['parent_id'] for r in RANK['MRTGAN_s12']][:K]
SCENES=P+[i for i in bba.digest_order(ripe_ids) if i not in P]
FRUITS=bba.digest_order([r['id'] for r in pool_manifest['records'] if r['domain']=='A'])
assert len(FRUITS)==81 and len(SCENES)==217
PR=WORK/'paste_real'; (PR/'images').mkdir(parents=True); (PR/'labels').mkdir()
PASTE_LOG=[]
for i,scene in enumerate(SCENES[:44]):
    row=rows[scene]; im=np.asarray(Image.open(DATASET/row['image']).convert('RGB')); H,W=im.shape[:2]
    box=yolo_box((DATASET/row['label']).read_text(),W,H)
    rm=(np.asarray(Image.open(POOL512/byid[scene]['mask']).convert('L'))>127).astype(np.uint8)
    clean=remove_fruit(im,scene_mask(byid[scene],rm,H,W),box)
    f=byid[FRUITS[i%81]]
    crop=np.asarray(Image.open(POOL512/f['image']).convert('RGB')); mask=(np.asarray(Image.open(POOL512/f['mask']).convert('L'))>127).astype(np.uint8)
    x,y,w,h=f['crop_box']; out,nb=paste(crop,mask,[int(x),int(y),int(x+w),int(y+h)],clean,box)
    Image.fromarray(out).save(PR/'images'/f'{scene}.jpg',quality=95); (PR/'labels'/f'{scene}.txt').write_text(to_yolo(1,nb,W,H))
    PASTE_LOG.append(dict(scene=scene,fruit=FRUITS[i%81]))
ARMLIST=['real_only','oversample_k22','colour_transfer_k22','mrtgan_k22']
def dup_for(k): return [(f"{order[i%len(order)]}__dup{i//len(order)}",DATASET/rows[order[i%len(order)]]['image'],(DATASET/rows[order[i%len(order)]]['label']).read_text(),order[i%len(order)]) for i in range(k)]
def ranked(arm,folder,k): return [(f'add_{i:04d}',folder/'full/images'/f"balance__{r['parent_id']}.jpg",(folder/'full/labels'/f"balance__{r['parent_id']}.txt").read_text(),r['parent_id']) for i,r in enumerate(RANK[arm][:k])]
def pasted(k): return [(f'add_{i:04d}',PR/'images'/f"{SCENES[i]}.jpg",(PR/'labels'/f"{SCENES[i]}.txt").read_text(),SCENES[i]) for i in range(k)]
for name,add in ((f'oversample_k11',dup_for(11)),('oversample_k44',dup_for(44)),
                 ('colour_transfer_k11',ranked('colour_transfer',ct,11)),('colour_transfer_k44',ranked('colour_transfer',ct,44)),
                 ('mrtgan_k11',ranked('MRTGAN_s12',CAND/'MRTGAN_s12',11)),
                 ('paste_real_k11',pasted(11)),('paste_real_k22',pasted(22)),('paste_real_k44',pasted(44))):
    ARM_INFO[name]=dict(bba.write_arm(ARMS,name,allrows,DATASET,add,excl)['counts'],added=len(add)); ARMLIST.append(name)
print(ARMLIST,len(ARMLIST))
assert K==22 and len(ARMLIST)==12
# Pre-letterbox train images (identity checked on real_only): the locked trainer otherwise decodes a 2048 px JPEG at every update.
import importlib.util, shutil as _sh
TR=cfg.BENCHMARK/'tools/train_detector.py'
spec=importlib.util.spec_from_file_location('train_detector',TR); trainer=importlib.util.module_from_spec(spec); sys.modules['train_detector']=trainer; spec.loader.exec_module(trainer)
ARMS640=WORK/'arms640'
def plb(name):
    arm,out=ARMS/name,ARMS640/name
    for split in ('train','val'):
        (out/'images'/split).mkdir(parents=True); (out/'labels'/split).mkdir(parents=True)
    ds=trainer.ArmDataset(arm,'train')
    for i in range(len(ds)):
        stem,image,boxes=ds.load(i,flip=False)
        image.save(out/'images/train'/f'{stem}.png')
        (out/'labels/train'/f'{stem}.txt').write_text(''.join(f'{c_} {x!r} {y!r} {w!r} {h!r}\n' for c_,x,y,w,h in boxes))
    for sub in ('images','labels'):
        for p in sorted((arm/sub/'val').glob('*')): _sh.copyfile(p,out/sub/'val'/p.name)
    _sh.copy(arm/'arm_manifest.json',out/'arm_manifest.json')
    (out/'data.yaml').write_text((arm/'data.yaml').read_text().replace(f'path: {arm.resolve()}',f'path: {out.resolve()}'))
for name in ARMLIST: plb(name)
a=trainer.ArmDataset(ARMS/'real_only','train'); b=trainer.ArmDataset(ARMS640/'real_only','train')
assert [p.stem for p in a.items]==[p.stem for p in b.items]
for i in range(0,len(a),10):
    for fl in (False,True):
        assert np.array_equal(np.asarray(a.load(i,fl)[1]),np.asarray(b.load(i,fl)[1]))
print('pre-letterbox identity ok')


In [ ]:
# Locked trainer, seeds 0-4. Twelve arms in four chains, two per T4.
import shlex, time
PLAN={i:ARMLIST[i::4] for i in range(4)}
n_gpu=torch.cuda.device_count()
def chain(i,arms):
    argv=[sys.executable,str(HERE/'tools/train_detector_arms.py'),'--arms-dir',str(ARMS640),'--arms',*arms,
          '--seeds','0','1','2','3','4','--out',str(WORK/f'group{i}'),'--updates','3000','--eval-every','250','--export']
    env=f"CUDA_VISIBLE_DEVICES={i%max(n_gpu,1)} OMP_NUM_THREADS=1 PYTHONPATH={shlex.quote(str(HERE))}"
    return subprocess.Popen(env+' '+shlex.join(argv),shell=True,cwd=str(HERE),stdout=open(WORK/f'detector_chain{i}.log','w'),stderr=subprocess.STDOUT)
started=time.time()
if n_gpu<1: raise SystemExit('GPU required')
codes=[p.wait() for p in [chain(i,PLAN[i]) for i in PLAN]]
WALL=round((time.time()-started)/60,1); print('return codes',codes,'wall minutes',WALL,flush=True)
for g in PLAN: print((WORK/f'detector_chain{g}.log').read_text()[-800:])
if any(codes): raise SystemExit('detector training failed')
RESULTS,SELECTED={},{}
for g in PLAN:
    part=json.loads((WORK/f'group{g}'/'results.json').read_text()); RESULTS.update(part['results']); SELECTED.update(part['selected'])
missing=[f'{a}@{sd}' for a in ARMLIST for sd in range(5) if f'{a}@{sd}' not in SELECTED]
if missing: raise SystemExit(f'missing runs {missing}')
PROBE={}


In [ ]:
# Results, small artefacts and ONE zip for download (per-file kernel output download is too slow).
import statistics, zipfile
OUT=WORK/'result'; OUT.mkdir()
def summ(table,arm):
    keys=[f'{arm}@{sd}' for sd in range(5)]
    return dict(map50_95=statistics.fmean(table[k]['map50_95'] for k in keys),sd=statistics.pstdev(table[k]['map50_95'] for k in keys),
                ap_unripe=statistics.fmean(table[k]['per_class']['unripe']['ap50_95'] for k in keys),
                per_seed=[table[k]['map50_95'] for k in keys])
arms=ARMLIST
report=dict(arms=arms,seeds=[0,1,2,3,4],updates=3000,eval_every=250,split='valid',test_used=False,wall_minutes_training=WALL,
            review_sha256=REVIEW_SHA,acceptance={a:dict(accepted=sum(v.values()),total=len(v)) for a,v in ACC.items()},
            integrity=INTEGRITY,pool512=pool_check,deficit=TARGET,count_matched_additions=K,arm_counts=ARM_INFO,ranking_reference=REF,
            summary_selected={a:summ(SELECTED,a) for a in arms},summary_final={a:summ(RESULTS,a) for a in arms},
            results_selected=SELECTED,results_final=RESULTS,probe=PROBE)
(OUT/'results.json').write_text(json.dumps(report,indent=1,default=str))
for g in PLAN: shutil.copytree(WORK/f'group{g}',OUT/f'group{g}',ignore=shutil.ignore_patterns('last*.pt'))
for arm in ('MRTGAN_s12','colour_transfer'):
    shutil.copy(WORK/f'ranking_{arm}.csv',OUT/f'ranking_{arm}.csv')
for name in ARM_INFO: shutil.copy(ARMS/name/'arm_manifest.json',OUT/f'arm_manifest_{name}.json')
with zipfile.ZipFile(WORK/'exposure_controls_result.zip','w',zipfile.ZIP_STORED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file(): z.write(p,p.relative_to(WORK))
(WORK/'exposure_controls_result.sha256').write_text(hashlib.sha256((WORK/'exposure_controls_result.zip').read_bytes()).hexdigest()+'\n')
(WORK/'paste_log.json').write_text(json.dumps(PASTE_LOG,indent=1))
for d in (STUDY,ARMS,ARMS640,PR,CAND,WORK/'code',OUT): shutil.rmtree(d,ignore_errors=True)
for g in PLAN: shutil.rmtree(WORK/f'group{g}',ignore_errors=True)
print(json.dumps(report['summary_selected'],indent=1)); print('zip MB',round((WORK/'exposure_controls_result.zip').stat().st_size/2**20))
